<a href="https://colab.research.google.com/github/CynthiahL/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

duka = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Shape:", duka.shape)
print("\nData Types:\n", duka.dtypes)
print("\nMissing Values:\n", duka.isna().sum())
print("\nDate Range:", duka["order_date"].min(), "to", duka["order_date"].max())
print("\nBranches:", duka["branch"].unique())
print("Products:", duka["product"].unique())
print("Channels:", duka["channel"].unique())


Shape: (1000, 7)

Data Types:
 order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object

Missing Values:
 order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Date Range: 2026-01-01 00:00:00 to 2026-06-29 00:00:00

Branches: ['Nakuru' 'Nairobi' 'Mombasa' 'Eldoret' 'Kisumu']
Products: ['Phone' 'Laptop' 'Tablet' 'Headphones' 'Charger']
Channels: ['In-store' 'Online' 'Phone order']


In [4]:
duka["revenue"] = duka["quantity"] * duka["unit_price"]
duka["month"] = duka["order_date"].dt.to_period("M")
duka["weekday"] = duka["order_date"].dt.day_name()
duka["is_weekend"] = duka["order_date"].dt.dayofweek >= 5
duka["price_band"] = duka["unit_price"].apply(lambda x: "Premium" if x >= 28000 else "Value")


In [5]:
# Task 3: Rank Branches by Total Revenue vs. Order Count

branch_summary = (
    duka.groupby("branch", as_index=False)
    .agg(total_revenue=("revenue", "sum"), order_count=("order_id", "count"))
)
branch_summary["rev_rank"] = branch_summary["total_revenue"].rank(ascending=False).astype(int)
branch_summary["order_rank"] = branch_summary["order_count"].rank(ascending=False).astype(int)
print(branch_summary.sort_values("total_revenue", ascending=False))

'''
Ans:
    branch  total_revenue  order_count  rev_rank  order_rank
3  Nairobi       26106100          376         1           1
2  Mombasa       14003700          186         2           2
4   Nakuru       11970400          162         3           3
1   Kisumu       10868100          159         4           4
0  Eldoret        8844200          117         5           5
'''

    branch  total_revenue  order_count  rev_rank  order_rank
3  Nairobi       26106100          376         1           1
2  Mombasa       14003700          186         2           2
4   Nakuru       11970400          162         3           3
1   Kisumu       10868100          159         4           4
0  Eldoret        8844200          117         5           5


In [6]:
#Task 4: Top 3 Products by Revenue vs. Units Sold
prod_summary = (
    duka.groupby("product", as_index=False)
    .agg(total_revenue=("revenue", "sum"), units_sold=("quantity", "sum"))
)
print("Top 3 by Revenue:\n", prod_summary.sort_values("total_revenue", ascending=False).head(3))
print("\nTop 3 by Units Sold:\n", prod_summary.sort_values("units_sold", ascending=False).head(3))

''' Ans:
Top 3 by Revenue:
   product  total_revenue  units_sold
2  Laptop       41535000         639
4  Tablet       19348000         691
3   Phone        8880000         592

Top 3 by Units Sold:
   product  total_revenue  units_sold
4  Tablet       19348000         691
2  Laptop       41535000         639
3   Phone        8880000         592
'''

Top 3 by Revenue:
   product  total_revenue  units_sold
2  Laptop       41535000         639
4  Tablet       19348000         691
3   Phone        8880000         592

Top 3 by Units Sold:
   product  total_revenue  units_sold
4  Tablet       19348000         691
2  Laptop       41535000         639
3   Phone        8880000         592


In [7]:
# Task 5: Weekend vs. Weekday Orders

weekend_comp = (
    duka.groupby("is_weekend", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean"),
    )
)
print(weekend_comp)

''' Ans:
  is_weekend  order_count  total_revenue  avg_order_revenue
0       False          723       51856200       71723.651452
1        True          277       19936300       71972.202166
'''

   is_weekend  order_count  total_revenue  avg_order_revenue
0       False          723       51856200       71723.651452
1        True          277       19936300       71972.202166


In [8]:
#Task 6: Best vs. Worst Month for Revenue
monthly_rev = duka.groupby("month")["revenue"].sum().sort_values(ascending=False)
best_month, best_val = monthly_rev.index[0], monthly_rev.iloc[0]
worst_month, worst_val = monthly_rev.index[-1], monthly_rev.iloc[-1]
pct_diff = ((best_val - worst_val) / worst_val) * 100

print(monthly_rev)
print(f"\nBest: {best_month} (KES {best_val:,.0f}) | Worst: {worst_month} (KES {worst_val:,.0f})")
print(f"Percentage difference: {pct_diff:.2f}%")

'''
month
2026-04    13518300
2026-03    12396800
2026-01    11786600
2026-02    11772000
2026-05    11415400
2026-06    10903400
Freq: M, Name: revenue, dtype: int64

Best: 2026-04 (KES 13,518,300) | Worst: 2026-06 (KES 10,903,400)
Percentage difference: 23.98%
'''

month
2026-04    13518300
2026-03    12396800
2026-01    11786600
2026-02    11772000
2026-05    11415400
2026-06    10903400
Freq: M, Name: revenue, dtype: int64

Best: 2026-04 (KES 13,518,300) | Worst: 2026-06 (KES 10,903,400)
Percentage difference: 23.98%


In [9]:
#Task 7: Filter & Export Premium Online Coast/Capital Orders
import os
os.makedirs("outputs", exist_ok=True)

filtered = duka[
    (duka["price_band"] == "Premium")
    & (duka["channel"] == "Online")
    & (duka["branch"].isin(["Nairobi", "Mombasa"]))
]
print("Order Count:", len(filtered), "| Total Revenue: KES", filtered["revenue"].sum())
filtered.to_csv("outputs/premium_online_coast_capital.csv", index=False)

# Ans: Order Count: 83 | Total Revenue: KES 10218000


Order Count: 83 | Total Revenue: KES 10218000


In [ ]:

'''
# Duka Electronics — Half-Year Sales Exploration (Jan–Jun 2026)

## Executive Summary
* Duka Electronics generated a total half-year revenue of **KES 71,792,500** across **1,000 orders** between January 1 and June 29, 2026.
* **Nairobi** led all branches in both revenue (**KES 26,106,100**, or 36.4% of total sales) and order volume (**376 orders**), followed by **Mombasa** (**KES 14,003,700** across 186 orders).
* **Laptops** generated the highest revenue at **KES 41,535,000** (639 units sold), whereas **Tablets** led total unit volume with **691 units** sold (**KES 19,348,000**).
* Peak monthly revenue occurred in **April 2026** (**KES 13,518,300**), outperforming the lowest month, **June 2026** (**KES 10,903,400**), by **23.98%**.
* High-value **Premium Online orders** from Nairobi and Mombasa accounted for **83 orders** generating **KES 10,218,000** in revenue.
'''

'\n# Duka Electronics — Half-Year Sales Exploration (Jan–Jun 2026)\n\n## Executive Summary\n* Duka Electronics generated a total half-year revenue of **KES 71,792,500** across **1,000 orders** between January 1 and June 29, 2026.\n* **Nairobi** led all branches in both revenue (**KES 26,106,100**, or 36.4% of total sales) and order volume (**376 orders**), followed by **Mombasa** (**KES 14,003,700** across 186 orders).\n* **Laptops** generated the highest revenue at **KES 41,535,000** (639 units sold), whereas **Tablets** led total unit volume with **691 units** sold (**KES 19,348,000**).\n* Peak monthly revenue occurred in **April 2026** (**KES 13,518,300**), outperforming the lowest month, **June 2026** (**KES 10,903,400**), by **23.98%**.\n* High-value **Premium Online orders** from Nairobi and Mombasa accounted for **83 orders** generating **KES 10,218,000** in revenue.\n'

In [12]:
import os
os.makedirs("excel", exist_ok=True)
duka.to_excel("excel/duka_orders_base.xlsx", sheet_name="Orders", index=False)
from google.colab import files

files.download("excel/duka_orders_base.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
xl_summary = pd.read_excel("excel/duka_branch_summary.xlsx", sheet_name="Branch_Summary")
xl_total = xl_summary.loc[xl_summary["Branch"] != "Total", "Total Revenue"].sum()
pd_total = duka["revenue"].sum()

print(f"Excel SUMIFS Total : KES {xl_total:,.0f}")
print(f"Pandas Total       : KES {pd_total:,.0f}")
assert xl_total == pd_total, "Totals do not match!"
print("Verified: Excel totals match pandas 100%.")


Excel SUMIFS Total : KES 71,792,500
Pandas Total       : KES 71,792,500
Verified: Excel totals match pandas 100%.
